# Install packages

In [0]:
!pip install duckdb

# Import packages

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

from scipy.stats import kurtosis, zscore, skew, mode, pearsonr
import math
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import duckdb

In [0]:
import warnings
warnings.filterwarnings('ignore')

In [0]:
df_data = pd.read_csv('prueba.csv', sep=";").drop('Unnamed: 0', axis =1 )

In [0]:
df_data

# Data exploration

In [0]:
df_data.APLICACION.value_counts().sort_index()

In [0]:
#define a function for clear aplicacion descriptions

def clear_descriptions(aplicacion): 
    if aplicacion in ['Spotifyy','Spotify_']:
        return 'Spotify'
    elif aplicacion == 'Netflix_':
        return 'Netflix'
    elif aplicacion =='Shazammm':
        return 'Shazam'
    elif aplicacion == 'Apple_Music_':
        return 'Apple_Music'
    else:
        return aplicacion    

In [0]:
df_data['APLICACION'] = df_data['APLICACION'].apply(clear_descriptions)

In [0]:
df_data.APLICACION.value_counts().sort_index()

In [0]:
df_data.APLICACION.value_counts()

In [0]:
df_data = df_data[df_data.APLICACION != 'TNT']

In [0]:
df_data.APLICACION.value_counts()/df_data.shape[0]

In [0]:
df_data.isna().sum().sort_values()

In [0]:
df_data = df_data.dropna()

In [0]:
df_data.isna().sum().sort_values()

In [0]:
df_data

In [0]:
df_data.APLICACION_CATEGORIA.value_counts()

In [0]:
df_data.head()

In [0]:
df_data[df_data.TIEMPO_CONEXION<=0]

In [0]:
df_data[df_data.TRAFICO<=0]

In [0]:
df_data = df_data[df_data.TRAFICO>0]

In [0]:
df_data.EDAD.value_counts().sort_index().tail(50)

In [0]:
df_data.EDAD.value_counts().sort_index().head(50)

In [0]:
df_data = df_data[df_data.EDAD.between(18,90)]

In [0]:
df_data.head()

In [0]:
df_data.shape

In [0]:
# graph box plot for each application by trafic

plt.figure(figsize = (15,8))
ax = sns.boxplot(x = 'APLICACION', y = 'TRAFICO',data = df_data, log_scale=True)
ax.set_xticklabels(ax.get_xticklabels(),rotation = 90)
plt.title('Comparacion de s de trafico para cada aplicacion en escala logaritmica')
plt.show()

In [0]:
# graph box plot for each application by trafic

plt.figure(figsize = (15,8))
ax = sns.boxplot(x = 'APLICACION', y = 'TIEMPO_CONEXION',data = df_data, log_scale=True)
ax.set_xticklabels(ax.get_xticklabels(),rotation = 90)
plt.title('Comparacion de tiempo de conexion para cada aplicacion en escala logaritmica')
plt.show()

In [0]:
# graph box plot for each application by trafic

plt.figure(figsize = (15,8))
ax = sns.boxplot(x = 'APLICACION', y = 'EDAD',data = df_data, log_scale=False)
ax.set_xticklabels(ax.get_xticklabels(),rotation = 90)
plt.title('Comparacion de edad para cada aplicacion')
plt.show()

In [0]:
query_  = f'''
    SELECT
        USUARIO,
        avg(EDAD) as age, 
        count(distinct APLICACION_ID) as nunique_apps,
        sum(case when APLICACION_CATEGORIA = 'VIDEO STREAMING' then TRAFICO end)/1000000000 as sum_video_streaming_gb,
        sum(case when APLICACION_CATEGORIA = 'MUSIC STREAMING' then TRAFICO end)/1000000000 as sum_music_streaming_gb,
        sum(case when APLICACION_CATEGORIA = 'VIDEO STREAMING' then TIEMPO_CONEXION end)/60 as sum_video_streaming_min,
        sum(case when APLICACION_CATEGORIA = 'MUSIC STREAMING' then TIEMPO_CONEXION end)/60 as sum_music_streaming_min,
        avg(TIEMPO_CONEXION)/60 as avg_min_time,
        avg(TRAFICO)/1000000 as avg_gb_traffic
    from
         df_data
    group by USUARIO
'''

df_group_data = duckdb.query(query_).to_df().fillna(0)

In [0]:
df_group_data

In [0]:
df_group_data.groupby('nunique_apps').agg({'age':'mean'})

In [0]:
df_group_data.groupby('nunique_apps').agg({'age':'mean'}).plot()

In [0]:
df_group_data.groupby('nunique_apps').agg({'sum_video_streaming_gb':'mean'}).plot()

In [0]:
df_group_data.groupby('nunique_apps').agg({'sum_music_streaming_gb':'mean'}).plot()

In [0]:
df_group_data['bracket_age'] = pd.qcut(df_group_data.age, 5)

In [0]:
df_group_data.head()

In [0]:
df_group_data.groupby('bracket_age').agg({'avg_min_time':'mean'}).plot()

In [0]:
df_group_data.groupby('bracket_age').agg({'avg_gb_traffic':'mean'}).plot()

In [0]:
import seaborn as sns

In [0]:

sns.histplot(data = df_group_data, x= 'sum_video_streaming_gb')

In [0]:
df_group_data['log_sum_video_streaming_gb'] = df_group_data['sum_video_streaming_gb'].apply(lambda x: np.log(x + 1))

In [0]:
sns.histplot(data = df_group_data, x= 'log_sum_video_streaming_gb')

In [0]:
df_group_data.head()

In [0]:
df_group_data['log_sum_music_streaming_gb'] = df_group_data['sum_music_streaming_gb'].apply(lambda x: np.log(x + 1))

In [0]:
df_group_data['log_avg_gb_traffic'] = df_group_data['avg_gb_traffic'].apply(lambda x: np.log(x + 1))

In [0]:
df_group_data

In [0]:
df_group_data.set_index('USUARIO', inplace=True)

In [0]:
plt.figure(figsize = (16,8))
ax = sns.heatmap(df_group_data.select_dtypes(include = np.number).corr(), annot=True)
plt.show()

In [0]:
df_group_data.head()

In [0]:
list_vars = [
    'age',
    'nunique_apps',
    'log_sum_music_streaming_gb',
    'log_sum_video_streaming_gb',
    'sum_video_streaming_min',
    'sum_music_streaming_min',
    'avg_min_time'
]

In [0]:
plt.figure(figsize = (16,8))
ax = sns.heatmap(df_group_data[list_vars].corr(), annot=True)
plt.show()

# Prepare data

In [0]:
df_group_data[list_vars].describe()

In [0]:
df_to_kmeans = df_group_data[list_vars].reset_index()

In [0]:
df_to_kmeans

In [0]:
from pyspark.sql import SparkSession

In [0]:
# create session

spark = SparkSession.builder.appName("ML").getOrCreate()

In [0]:
df_spark = spark.createDataFrame(df_to_kmeans.drop('USUARIO', axis = 1).sample(frac = 0.5))

In [0]:
df_spark.show(5)

# Train model

In [0]:

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [0]:
list_vars 

In [0]:
assembler = VectorAssembler(
    inputCols = list_vars,
    outputCol = "features_raw"
)


In [0]:
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [0]:
X = df_to_kmeans[list_vars]

In [0]:
X

In [0]:
X_standard = StandardScaler().fit_transform(X)

In [0]:
inertias = []
K = range(1, 10)
  
for k in K:
    # building and fitting the model

    kmeanModel = KMeans(n_clusters=k, init = 'k-means++', random_state=42)
    kmeanModel.fit(X)
    inertias.append(kmeanModel.inertia_)

In [0]:
#the plot suggest to use four clusters for this case

plt.plot(K, inertias, 'bx-')
plt.xlabel('Values of K')
plt.ylabel('Inertias')
plt.title('The Elbow Method using Inertias')
plt.show()

In [0]:
# assing clusters group

clustering = KMeans(n_clusters=4, init = 'k-means++',random_state=42) 
clustering.fit(X)
df_to_kmeans["CLASIFICATION"] = clustering.labels_

In [0]:
df_to_kmeans.head()

In [0]:
df_to_kmeans.CLASIFICATION.value_counts()

In [0]:
def plot_radar_provisional_generic(df_sample, list_names, colors):

    categories = df_sample.columns.tolist()
    categories = [c.replace('x', '\n') for c in categories] 
    N = len(categories)

    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

    for i, color in enumerate(colors):
        row = df_sample.iloc[i].tolist()
        row += row[:1]

        ax.fill(angles, row, color=color, alpha=0.25)
        ax.plot(angles, row, color=color, linewidth=2, label=f'{list_names[i]}')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=15, ha='center', va='center')
    ax.tick_params(axis='x', pad=15)  # separa las etiquetas del centro
    ax.set_yticklabels([])

    plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1), fontsize=14)
    plt.show()
    # plt.savefig(path_,bbox_inches="tight")
    # plt.close()

In [0]:
import numpy as np
import matplotlib.pyplot as plt

In [0]:
df_to_kmeans.set_index('CLASIFICATION', inplace = True)

In [0]:
df_to_kmeans


In [0]:
df_to_kmeans.select_dtypes(include = np.number).groupby(level = 0).mean().drop('USUARIO', axis = 1)

# Radar plot

In [0]:
from sklearn.preprocessing import MinMaxScaler

In [0]:
scaler = MinMaxScaler()

In [0]:
df_to_plot_= pd.DataFrame(scaler.fit_transform(df_to_kmeans), index = df_to_kmeans.index, columns = df_to_kmeans.columns).drop('USUARIO', axis = 1)

In [0]:
df_to_plot_refine = df_to_plot_.select_dtypes(include = np.number).groupby(level = 0).mean()

In [0]:
df_to_plot_refine

In [0]:
plot_radar_provisional_generic(np.log(df_to_plot_refine), df_to_plot_refine.index.tolist(), ['lightcoral','blue','black','green'])